# 01. LangGraph를 위한 Python 기초

> LangGraph 코드는 `TypedDict` · `Annotated` · `add_messages` 같은 Python 타이핑 기능을 적극 사용해요. 본격적인 그래프 학습 전에 필요한 Python 기초만 모아 빠르게 점검해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `TypedDict`와 일반 `dict`의 차이를 설명하고, LangGraph에서 State를 정의할 때 TypedDict를 사용하는 이유를 이해할 수 있어요
2. `Annotated`를 사용하여 타입 힌트에 메타데이터를 붙이고, LangGraph의 리듀서(reducer)와 연결하는 방법을 알 수 있어요
3. Pydantic `BaseModel`과 `Field`로 데이터 유효성 검사(validation)를 구현할 수 있어요
4. `add_messages` 리듀서의 동작 원리(추가 vs. 교체)를 이해하고 직접 사용할 수 있어요

## 사전 지식

- Python 딕셔너리(`dict`)와 클래스(`class`) 기본 문법
- 이전 노트북: `01_Introduction/02-Product-Hierarchy.ipynb` — LangChain vs LangGraph vs Deep Agents 제품 계층


## 왜 이 문법을 배워야 할까요?

LangGraph로 에이전트를 만들 때 우리는 반드시 **State(상태)**를 정의해야 해요. 이 State는 그래프가 실행되는 동안 노드 간에 데이터를 전달하는 역할을 해요.

State를 올바르게 정의하려면 세 가지 Python 도구를 알아야 해요:

| 도구 | 역할 | LangGraph에서 사용 위치 |
|------|------|------------------------|
| `TypedDict` | 딕셔너리의 키/타입 구조를 고정 | State 클래스 정의 |
| `Annotated` | 타입에 메타데이터(리듀서) 첨부 | `messages: Annotated[list, add_messages]` |
| `Pydantic BaseModel` | 입력 데이터 유효성 검사 | 노드의 입력/출력 스키마 |
| `add_messages` | 메시지 리스트를 스마트하게 병합 | State의 messages 필드 리듀서 |

아래 다이어그램에서 이 도구들이 LangGraph 안에서 어떻게 연결되는지 살펴볼게요:


```mermaid
flowchart TD
    A["Python TypedDict<br>상태 구조 정의"] --> D
    B["Python Annotated<br>리듀서 메타데이터 첨부"] --> D
    C["Pydantic BaseModel<br>입력값 유효성 검사"] --> E
    D["LangGraph State<br>class State(TypedDict):<br>  messages: Annotated[list, add_messages]"] --> F
    E["노드 입력/출력 스키마"] --> F
    F["StateGraph 컴파일 및 실행"]
    G["add_messages 리듀서<br>메시지 추가 또는 교체"] --> D

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class A,B,C,G input
    class D,E process
    class F output
```


---

## 1. TypedDict — 구조가 있는 딕셔너리

Python의 일반 `dict`는 어떤 키든 자유롭게 추가/변경할 수 있어요. 이건 편리하지만, 코드가 복잡해지면 실수가 생기기 쉬워요.

`TypedDict`는 **딕셔너리의 키와 각 값의 타입을 미리 선언**하는 방법이에요. 코드 작성 시점에 IDE가 오류를 잡아줄 수 있어요.

> 🔑 **핵심 개념**: TypedDict는 런타임(실행 중)에는 일반 dict와 동일하게 동작해요. 차이는 **정적 타입 검사** 시점(코드 작성 중, IDE)에 있어요. LangGraph는 TypedDict를 State 정의의 표준으로 사용해요.

비유하자면, 일반 `dict`는 **아무 물건이나 넣을 수 있는 서랍장**이에요. TypedDict는 **칸마다 라벨이 붙은 정리함**이에요 — "양말 칸", "속옷 칸"처럼 각 칸에 뭘 넣을지 미리 정해놨어요. 실제로 넣을 때 강제하진 않지만, 라벨을 보고 실수를 줄일 수 있어요.


In [4]:
from typing import Dict, TypedDict

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: TypedDict vs dict 비교
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.
sample_dict: Dict[str, str] = {
    "name": "elixir",
    "age": "30",
    "job": "개발자",
}

sample_dict["age"] = 30
sample_dict["country"] = "Korea"

print(sample_dict)


{'name': 'elixir', 'age': 30, 'job': '개발자', 'country': 'Korea'}


In [5]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 2) TypedDict: 구조를 미리 선언
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.

class Person(TypedDict):
    name: str
    age: int
    job: str

typed_person: Person = {
    "name": "asd",
    "age": 30,
    "job": "개발자"
}

typed_person

{'name': 'asd', 'age': 30, 'job': '개발자'}

In [6]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: LangGraph에서 State 정의 패턴
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.
class AgentState(TypedDict):
    query: str
    answer: str
    step: int
    is_done: bool

state: AgentState = {
    "query": "LangGraph에 대해 설명해",
    "answer": "",
    "step": 0,
    "is_done": False
}

---

## 2. Annotated — 타입에 메타데이터 붙이기

### 왜 리듀서가 필요할까요?

LangGraph에서 여러 노드가 같은 State 필드를 수정해요. 예를 들어, 노드 A가 메시지를 추가하고, 노드 B도 메시지를 추가해요. 만약 단순 덮어쓰기라면 노드 B의 메시지만 남고 노드 A의 메시지는 사라져요! **리듀서(reducer)**는 "이 필드를 업데이트할 때 덮어쓰지 말고 이렇게 합쳐라"는 규칙을 정의해요. 그리고 이 규칙을 타입 힌트에 붙이는 도구가 바로 `Annotated`예요.

`Annotated`는 Python 3.9에서 도입된 타입 힌트 도구예요. **기존 타입에 추가 정보(메타데이터)**를 붙일 수 있게 해줘요.

기본 문법은 이렇게 생겼어요:
```python
변수명: Annotated[원래_타입, 메타데이터]
```

LangGraph에서 `Annotated`의 핵심 용도는 **리듀서(reducer) 함수를 State 필드에 연결**하는 거예요:
```python
messages: Annotated[list, add_messages]  # add_messages가 리듀서
```


In [ ]:
from typing import Annotated

import typing

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Annotated 기본 사용법
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.

class AgentState(TypedDict):
    name: Annotated[str, "사용자 이름(2~50자)"] = "elixir"

age_type = Annotated[int, "나이 (0~150 범위)"]
print(typing.get_args(age_type))


(<class 'int'>, '나이 (0~150 범위)')


In [22]:
from typing import TypedDict
from langgraph.graph.message import add_messages  # LangGraph 리듀서 함수

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: LangGraph에서 Annotated + 리듀서 패턴
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]

    user_name: str
    session_id: str

print(ChatState.__annotations__["messages"])

typing.Annotated[list, <function _add_messages_wrapper.<locals>._add_messages at 0x113f3bd70>]


In [29]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Annotated 없는 필드 vs. Annotated + 리듀서 필드 비교
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.
class OverwriteState(TypedDict):
    count: int

def append_reducer(left: list, right: list) -> list:
    return left + right

class AccumulateState(TypedDict):
    items: Annotated[list, append_reducer]

overwirte_state: OverwriteState = {"count": 1}
print(overwirte_state)

overwirte_state["count"] = 5

accumulate_state: AccumulateState = {"items": ["item1", "item2"]}
print(accumulate_state)

print(accumulate_state["items"])
accumulate_state["items"] = append_reducer(accumulate_state["items"], ["item3", "item4"])
print(accumulate_state["items"])

accumulate_state

{'count': 1}
{'items': ['item1', 'item2']}
['item1', 'item2']
['item1', 'item2', 'item3', 'item4']


{'items': ['item1', 'item2', 'item3', 'item4']}

---

## 3. Pydantic BaseModel — 데이터 유효성 검사

Pydantic은 Python 데이터 유효성 검사 라이브러리예요. LangGraph에서는 노드의 입력/출력 데이터나 도구(tool)의 파라미터를 정의할 때 많이 사용해요.

TypedDict가 **구조를 정의**한다면, Pydantic BaseModel은 **값의 제약 조건까지 검사**해요.

| 기능 | TypedDict | Pydantic BaseModel |
|------|-----------|--------------------|
| 키 타입 선언 | O | O |
| 값 제약 조건 (min, max 등) | X | O |
| 런타임 유효성 검사 | X | O |
| 자동 타입 변환 | X | O |
| LangGraph State 정의 | 주로 사용 | 가능 |


In [34]:
from pydantic import BaseModel, Field, ValidationError

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Pydantic BaseModel 기본 사용법 (V2 문법)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.
class Employee(BaseModel):
    """직원 정보 모음"""
    id: int = Field(description="직원의 고유 Id")
    name: str = Field(description="직원의 이름")
    age: int = Field(description="나이 19~64세", gt=18, lt=65)
    salary: float = Field(gt=0, description="월급")
    department: str = Field(default="일반", description="부서명")

emp = Employee(
    id=1,
    name="철수",
    age=30,
    salary=5000000,
    department="개발팀"
)
print(emp)

id=1 name='철수' age=30 salary=5000000.0 department='개발팀'


In [36]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 유효성 검사 실패 시 에러 처리
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.
try:
    invalid_emp = Employee(
        id=2,
        name="A",
        age=10,
        salary=-100
    )
except ValidationError as e:
    print(e)
    for error in e.errors():
        print(error)

2 validation errors for Employee
age
  Input should be greater than 18 [type=greater_than, input_value=10, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than
salary
  Input should be greater than 0 [type=greater_than, input_value=-100, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than
{'type': 'greater_than', 'loc': ('age',), 'msg': 'Input should be greater than 18', 'input': 10, 'ctx': {'gt': 18}, 'url': 'https://errors.pydantic.dev/2.13/v/greater_than'}
{'type': 'greater_than', 'loc': ('salary',), 'msg': 'Input should be greater than 0', 'input': -100, 'ctx': {'gt': 0.0}, 'url': 'https://errors.pydantic.dev/2.13/v/greater_than'}


In [37]:
from typing import Annotated
from pydantic import BaseModel, Field

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Pydantic + Annotated 조합 (V2 스타일)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.
# ---------------------------------------------------
# Pydantic + Annotated 조합 (V2 스타일)
# ---------------------------------------------------
# Pydantic V2에서는 Annotated와 Field를 조합하는 방식도 있어요
from typing import Annotated
from pydantic import BaseModel, Field

class SearchQuery(BaseModel):
    """LangGraph 도구에서 사용하는 검색 쿼리 스키마 예시"""
    query: Annotated[str, Field(min_length=1, description="검색 쿼리 문자열")]
    max_results: Annotated[int, Field(gt=0, le=10, default=5, description="최대 결과 수 (1~10)")]
    language: Annotated[str, Field(default="ko", description="검색 언어")]

# 기본값이 있는 필드는 생략 가능해요
search = SearchQuery(query="LangGraph 튜토리얼")
print("검색 쿼리:", search)
print("JSON 형태:", search.model_dump())
# (max_results, language는 기본값이 적용됐어요)

검색 쿼리: query='LangGraph 튜토리얼' max_results=5 language='ko'
JSON 형태: {'query': 'LangGraph 튜토리얼', 'max_results': 5, 'language': 'ko'}


---

## 4. add_messages 리듀서

`add_messages`는 LangGraph에서 제공하는 핵심 리듀서 함수예요. 채팅 애플리케이션에서 대화 기록을 관리할 때 사용해요.

### 동작 규칙

| 상황 | 동작 |
|------|------|
| 새 메시지 ID가 없거나 기존과 다름 | 기존 리스트에 **추가** |
| 새 메시지의 ID가 기존 메시지와 같음 | 기존 메시지를 **교체** |

이 규칙 덕분에 에이전트가 이전 메시지를 수정해야 할 때도 유연하게 처리할 수 있어요.

> 🔑 **핵심 개념**: `add_messages(left, right)`는 두 개의 메시지 리스트를 받아요. `left`는 현재 State의 messages, `right`는 노드가 반환한 새 messages예요. LangGraph가 자동으로 이 함수를 호출해서 상태를 업데이트해요.

비유하자면, `add_messages`는 **대화 녹음기**와 비슷해요. 새 발언이 들어오면 기존 녹음 뒤에 이어 붙이고(추가), 이미 녹음된 발언을 수정하고 싶으면 같은 ID의 발언을 새 버전으로 교체해요.


In [13]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph.message import add_messages

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: add_messages 기본 동작: 메시지 추가
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [14]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: add_messages 교체 동작: 같은 ID면 덮어써요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [15]:
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain.messages import HumanMessage, AIMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: add_messages를 실제 State에 적용한 예시
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 5. 종합 실습 — 모든 개념을 하나로

지금까지 배운 TypedDict, Annotated, Pydantic, add_messages를 모두 조합해서 실제 LangGraph State처럼 작성해볼게요.


In [16]:
from typing import Annotated, TypedDict
from pydantic import BaseModel, Field
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph.message import add_messages

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 종합: TypedDict + Annotated + add_messages 완전 패턴
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [17]:
# ============================================================
# TODO: 나만의 AgentState 만들어보기
# 아래 TODO 항목을 완성해보세요:
#
# 1. TypedDict로 TodoState를 정의하세요:
#    - messages: Annotated[list, add_messages] (메시지 누적)
#    - todos: Annotated[list, ???]             (할 일 목록 누적)
#    - current_task: str                       (현재 작업)
#
# 2. 할 일 목록용 리듀서 함수를 직접 만들어보세요:
#    - def todo_reducer(left: list, right: list) -> list:
#      중복 항목 없이 합치는 로직을 구현하면 좋아요
#
# 힌트: Annotated[list, 여기에_리듀서_함수_이름]
# 예상 결과: TodoState 인스턴스를 만들고 todos를 여러 번 업데이트해도
#            기존 할 일이 사라지지 않아야 해요
# ============================================================

# def todo_reducer(left: list, right: list) -> list:
#     """할 일 목록 리듀서: 중복 없이 합치기"""
#     # 여기에 구현하세요
#     pass

# class TodoState(TypedDict):
#     messages: Annotated[list, add_messages]
#     todos: Annotated[list, ???]
#     current_task: str

# 테스트
# state: TodoState = {"messages": [], "todos": [], "current_task": ""}
# state["todos"] = todo_reducer(state["todos"], ["LangGraph 공부", "실습 완료"])
# print("할 일 목록:", state["todos"])

# TODO 블록: 위의 주석을 해제하고 직접 구현해보세요!


---

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **TypedDict**: 딕셔너리의 키와 타입을 미리 선언해요. 런타임에는 일반 dict와 동일하지만, IDE와 타입 검사기가 오류를 미리 잡아줘요. LangGraph State 정의의 표준이에요.

- **Annotated**: 타입 힌트에 메타데이터를 붙이는 Python 도구예요. LangGraph에서는 `Annotated[list, add_messages]` 형태로 리듀서 함수를 State 필드에 연결할 때 사용해요.

- **Pydantic BaseModel**: 런타임 유효성 검사를 제공하는 라이브러리예요. `Field`로 최솟값, 최댓값, 설명 등을 지정할 수 있어요. V2에서는 `min_length`, `max_length`, `gt`, `lt` 파라미터를 사용해요.

- **add_messages**: LangGraph 내장 리듀서 함수예요. `(left, right)` 두 메시지 리스트를 받아서 병합해요. 같은 ID의 메시지가 있으면 교체, 없으면 추가해요. 대화 기록 누적에 필수예요.


## 다음 노트북 예고

다음 `02-Models.ipynb`에서는 **LangGraph에서 LLM 모델을 초기화하고 사용하는 방법**을 배워요. `init_chat_model`로 OpenAI, Anthropic, Google 등 다양한 모델을 통일된 인터페이스로 다루고, invoke/stream/batch 호출 패턴을 실습해요.
